# Spanner All-Storage Integration Test Notebook

This notebook tests PathRAG end-to-end with **all three** Spanner storage backends:

| Backend | Class | Used for |
|---------|-------|----------|
| Key-Value | `SpannerKVStorage` | `full_docs`, `text_chunks`, `llm_response_cache` |
| Vector | `SpannerVectorDBStorage` | `entities`, `relationships`, `chunks` |
| Graph | `SpannerGraphStorage` | `chunk_entity_relation` |

## Prerequisites

1. **Google Cloud authentication**
   ```bash
   gcloud auth application-default login
   ```

2. **Enable required APIs**
   ```bash
   gcloud services enable spanner.googleapis.com
   ```

3. **Create a Spanner instance and database** (if not already created)
   ```bash
   export SPANNER_INSTANCE="pathrag"
   export SPANNER_DATABASE="pathrag_db"
   export SPANNER_REGION="us-central1"

   gcloud spanner instances create $SPANNER_INSTANCE \
     --config=regional-$SPANNER_REGION \
     --description="Spanner instance for PathRAG" \
     --nodes=1 \
     --edition=ENTERPRISE

   gcloud spanner databases create $SPANNER_DATABASE \
     --instance=$SPANNER_INSTANCE
   ```

4. **Install dependencies**
   ```bash
   pip install -e .
   pip install google-cloud-spanner
   ```

5. **Configure `.env` file** in `examples/spanner/` (see `.env.example`)

> **Note:** All three storage backends automatically create their required tables and schemas on first initialization.

---
## Setup: Load Environment & Helpers

In [ ]:
import json
import os
import tempfile
import shutil

import numpy as np
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

SPANNER_INSTANCE = os.environ.get("SPANNER_INSTANCE")
SPANNER_DATABASE = os.environ.get("SPANNER_DATABASE")

print(f"Instance: {SPANNER_INSTANCE}")
print(f"Database: {SPANNER_DATABASE}")

assert SPANNER_INSTANCE and SPANNER_DATABASE, (
    "SPANNER_INSTANCE and SPANNER_DATABASE must be set in .env or environment variables.\n"
    "See examples/spanner/.env.example"
)

# Namespace prefix for integration test — avoids collision with individual tests
NS_PREFIX = "integ"

In [ ]:
from PathRAG.utils import EmbeddingFunc


def global_config(**overrides):
    """Build global_config dict for Spanner storage instances."""
    cfg = {
        "spanner_instance_id": SPANNER_INSTANCE,
        "spanner_database_id": SPANNER_DATABASE,
        "embedding_batch_num": 32,
    }
    cfg.update(overrides)
    return cfg


def create_mock_embedding_func(dim=128):
    """Create a deterministic mock embedding function for unit tests."""
    async def _mock_embed(texts: list[str]) -> np.ndarray:
        embeddings = []
        for text in texts:
            seed = sum(ord(c) for c in text) % (2**31)
            rng = np.random.RandomState(seed)
            vec = rng.randn(dim).astype(np.float64)
            vec = vec / np.linalg.norm(vec)
            embeddings.append(vec)
        return np.array(embeddings)

    return EmbeddingFunc(embedding_dim=dim, max_token_size=8192, func=_mock_embed)


print("Helpers ready.")

---
## Step 1: SpannerKVStorage Smoke Test

Test basic key-value operations: upsert, get, filter, drop.

In [ ]:
from PathRAG.storage.spanner import SpannerKVStorage

kv = SpannerKVStorage(
    namespace=f"{NS_PREFIX}_kv_smoke",
    global_config=global_config(),
    embedding_func=create_mock_embedding_func(),
)
print(f"SpannerKVStorage created (namespace={NS_PREFIX}_kv_smoke)")

### 1-1. Upsert

In [ ]:
data = {
    "key-1": {"content": "Hello World", "score": 1.0},
    "key-2": {"content": "Foo Bar", "score": 2.0},
}
left = await kv.upsert(data)
print(f"Upserted: {list(left.keys())}")

### 1-2. Read back

In [ ]:
val = await kv.get_by_id("key-1")
print(f"get_by_id('key-1'): {json.dumps(val)}")
assert val["content"] == "Hello World"

vals = await kv.get_by_ids(["key-1", "key-2", "missing"], fields={"content"})
print(f"get_by_ids (content only): {[json.dumps(v) if v else None for v in vals]}")
assert vals[2] is None

### 1-3. all_keys / filter_keys

In [ ]:
keys = await kv.all_keys()
print(f"all_keys: {keys}")
assert set(keys) == {"key-1", "key-2"}

missing = await kv.filter_keys(["key-1", "key-999"])
print(f"filter_keys(['key-1','key-999']): {missing}")
assert missing == {"key-999"}

### 1-4. Drop

In [ ]:
await kv.drop()
assert len(await kv.all_keys()) == 0
print("drop: OK (0 keys)")

print("\n[OK] KV Storage smoke test passed")

---
## Step 2: SpannerVectorDBStorage Smoke Test

Test vector upsert and similarity search.

In [ ]:
from PathRAG.storage.spanner import SpannerVectorDBStorage

embed_func = create_mock_embedding_func(dim=128)
vdb = SpannerVectorDBStorage(
    namespace=f"{NS_PREFIX}_vec_smoke",
    global_config=global_config(),
    embedding_func=embed_func,
    meta_fields={"entity_name"},
)
print(f"SpannerVectorDBStorage created (namespace={NS_PREFIX}_vec_smoke)")

### 2-1. Upsert vectors

In [ ]:
test_data = {
    "v-001": {"content": "Apple is a technology company.", "entity_name": "APPLE"},
    "v-002": {"content": "Google is a search engine.", "entity_name": "GOOGLE"},
    "v-003": {"content": "Microsoft makes Windows.", "entity_name": "MICROSOFT"},
}

print(f"Upserting {len(test_data)} vectors...")
await vdb.upsert(test_data)
print("Done.")

### 2-2. Query vectors

In [ ]:
results = await vdb.query("Apple technology", top_k=2)
print(f"Query 'Apple technology': {len(results)} results")
for r in results:
    print(f"  {r['id']}: entity={r.get('entity_name','N/A')}, dist={r['distance']:.4f}")

print("\n[OK] Vector Storage smoke test passed")

---
## Step 3: SpannerGraphStorage Smoke Test

Test node/edge CRUD and graph traversal.

In [ ]:
from PathRAG.storage.spanner import SpannerGraphStorage

graph = SpannerGraphStorage(
    namespace=f"{NS_PREFIX}_graph_smoke",
    global_config=global_config(),
)
print(f"SpannerGraphStorage created (namespace={NS_PREFIX}_graph_smoke)")

### 3-1. Insert nodes & edges

In [ ]:
await graph.upsert_node("NODE_A", {"entity_type": "ORG", "description": "Org A"})
await graph.upsert_node("NODE_B", {"entity_type": "PERSON", "description": "Person B"})
await graph.upsert_edge("NODE_B", "NODE_A", {
    "weight": "1.5", "description": "works at", "keywords": "employment",
})
print(f"Nodes: {await graph.nodes()}")
print(f"Edges: {await graph.edges()}")

### 3-2. Verify existence

In [ ]:
assert await graph.has_node("NODE_A")
assert await graph.has_edge("NODE_B", "NODE_A")
print("has_node / has_edge: OK")

### 3-3. Node degree & edge data

In [ ]:
degree = await graph.node_degree("NODE_A")
print(f"NODE_A degree: {degree}")

edge_data = await graph.get_edge("NODE_B", "NODE_A")
print(f"Edge B->A: {json.dumps(edge_data)}")

print("\n[OK] Graph Storage smoke test passed")

---
## Step 4: Full PathRAG Integration

End-to-end test with **all three Spanner backends** (KV + Vector + Graph) and a real LLM.

> **Requires:** `GEMINI_API_KEY` or `OPENAI_API_KEY` in `.env`

In [ ]:
def get_llm_config():
    """Resolve LLM configuration from environment variables."""
    if os.environ.get("GEMINI_API_KEY"):
        default_llm = "gemini/gemini-2.5-flash"
        default_embed = "gemini/gemini-embedding-001"
        default_dim = 3072
    elif os.environ.get("OPENAI_API_KEY"):
        default_llm = "gpt-4o-mini"
        default_embed = "text-embedding-3-small"
        default_dim = 1536
    else:
        print("[SKIP] No API key found (GEMINI_API_KEY or OPENAI_API_KEY).")
        return None, None, None, None

    llm_model = os.environ.get("LLM_MODEL_NAME", default_llm)
    embedding_model = os.environ.get("EMBEDDING_MODEL_NAME", default_embed)
    embedding_dim = int(os.environ.get("EMBEDDING_DIM", default_dim))
    tiktoken_model = llm_model

    print(f"LLM Model: {llm_model}")
    print(f"Embedding Model: {embedding_model} (dim={embedding_dim})")
    return llm_model, embedding_model, embedding_dim, tiktoken_model


llm_model, embedding_model, embedding_dim, tiktoken_model = get_llm_config()

### 4-1. Create PathRAG with all Spanner backends

In [ ]:
from PathRAG import PathRAG, QueryParam

work_dir = tempfile.mkdtemp(prefix="pathrag_spanner_integ_")
print(f"Working dir: {work_dir}")

rag = PathRAG(
    working_dir=work_dir,
    llm_model_name=llm_model,
    embedding_model_name=embedding_model,
    embedding_dim=embedding_dim,
    tiktoken_model_name=tiktoken_model,
    kv_storage="SpannerKVStorage",
    vector_storage="SpannerVectorDBStorage",
    graph_storage="SpannerGraphStorage",
    addon_params={
        "spanner_instance_id": SPANNER_INSTANCE,
        "spanner_database_id": SPANNER_DATABASE,
    },
    chunk_token_size=200,
    chunk_overlap_token_size=50,
    enable_llm_cache=False,
)

print("\n--- Storage backends ---")
print(f"  KV (full_docs):    {type(rag.full_docs).__name__}")
print(f"  KV (text_chunks):  {type(rag.text_chunks).__name__}")
print(f"  KV (llm_cache):    {type(rag.llm_response_cache).__name__}")
print(f"  Vector (entities): {type(rag.entities_vdb).__name__}")
print(f"  Vector (rels):     {type(rag.relationships_vdb).__name__}")
print(f"  Vector (chunks):   {type(rag.chunks_vdb).__name__}")
print(f"  Graph:             {type(rag.chunk_entity_relation_graph).__name__}")

assert "Spanner" in type(rag.full_docs).__name__
assert "Spanner" in type(rag.entities_vdb).__name__
assert "Spanner" in type(rag.chunk_entity_relation_graph).__name__
print("\nAll backends are Spanner-backed!")

### 4-2. Index documents

In [ ]:
sample_docs = [
    """
    Apple Inc. is an American multinational technology company
    headquartered in Cupertino, California. Apple was founded on
    April 1, 1976, by Steve Jobs, Steve Wozniak, and Ronald Wayne.
    Tim Cook has been the CEO of Apple since August 2011.
    """,
    """
    Google LLC is an American multinational corporation focusing on
    search engine technology and cloud computing. Sundar Pichai has
    been the CEO of Google since October 2015. Google was founded
    on September 4, 1998, by Larry Page and Sergey Brin.
    """,
]

print(f"Indexing {len(sample_docs)} documents...")
await rag.ainsert(sample_docs)
print("Indexing complete!")

### 4-3. Verify KV Storage

In [ ]:
full_doc_keys = await rag.full_docs.all_keys()
text_chunk_keys = await rag.text_chunks.all_keys()
print(f"full_docs keys: {len(full_doc_keys)}")
print(f"text_chunks keys: {len(text_chunk_keys)}")
assert len(full_doc_keys) > 0, "full_docs should have entries"
assert len(text_chunk_keys) > 0, "text_chunks should have entries"

if text_chunk_keys:
    sample_chunk = await rag.text_chunks.get_by_id(text_chunk_keys[0])
    print(f"Sample chunk content: {sample_chunk.get('content', '')[:80]}...")

### 4-4. Verify Graph Storage

In [ ]:
rag_graph = rag.chunk_entity_relation_graph
nodes = await rag_graph.nodes()
edges_list = await rag_graph.edges()
print(f"Entities: {len(nodes)}")
print(f"Relationships: {len(edges_list)}")

print("\n--- Entities ---")
for name in list(nodes)[:10]:
    data = await rag_graph.get_node(name)
    if data:
        print(f"  {name}: type={data.get('entity_type', 'N/A')}")

print("\n--- Relationships ---")
for src, tgt in list(edges_list)[:10]:
    data = await rag_graph.get_edge(src, tgt)
    if data:
        print(f"  {src} --[{data.get('keywords', 'N/A')}]--> {tgt}")

### 4-5. Verify Vector Storage

In [ ]:
entity_results = await rag.entities_vdb.query("Apple technology company", top_k=5)
print(f"Entity search 'Apple technology company': {len(entity_results)} results")
for r in entity_results:
    print(f"  entity={r.get('entity_name','N/A')}, dist={r['distance']:.4f}")

rel_results = await rag.relationships_vdb.query("founded the company", top_k=5)
print(f"\nRelationship search 'founded the company': {len(rel_results)} results")
for r in rel_results:
    print(f"  {r.get('src_id','N/A')} -> {r.get('tgt_id','N/A')}, dist={r['distance']:.4f}")

### 4-6. RAG Queries

In [ ]:
queries = [
    "What is the relationship between Steve Jobs and Apple?",
    "Who founded Google and when?",
    "Compare the leadership of Apple and Google.",
]

for q in queries:
    print(f"\nQ: {q}")
    response = await rag.aquery(
        q, param=QueryParam(mode="hybrid", top_k=10),
    )
    display = response[:300] + "..." if len(response) > 300 else response
    print(f"A: {display}")

print("\n[OK] Full PathRAG Integration test passed")

---
## Step 5: Cleanup All Test Resources

Drop all test tables and property graphs created during integration tests.

> **Warning:** This will permanently delete all test data. Only run this when you are done testing.

In [ ]:
from google.cloud import spanner

client = spanner.Client(disable_builtin_metrics=True)
instance = client.instance(SPANNER_INSTANCE)
database = instance.database(SPANNER_DATABASE)

# Property graphs (must be dropped before tables they reference)
graphs_to_drop = [
    f"pathrag_{NS_PREFIX}_graph_smoke",
    "pathrag_chunk_entity_relation",
]
for g in graphs_to_drop:
    print(f"Dropping property graph '{g}'...")
    try:
        op = database.update_ddl([f"DROP PROPERTY GRAPH IF EXISTS {g}"])
        op.result()
        print(f"  Done.")
    except Exception as e:
        print(f"  Skipped: {e}")

# Tables (order: edges before nodes due to FK constraints)
tables_to_drop = [
    # Smoke test tables
    f"{NS_PREFIX}_kv_smoke_kv",
    f"vdb_{NS_PREFIX}_vec_smoke",
    f"{NS_PREFIX}_graph_smoke_edges",
    f"{NS_PREFIX}_graph_smoke_nodes",
    # PathRAG integration tables — KV
    "full_docs_kv",
    "text_chunks_kv",
    "llm_response_cache_kv",
    # PathRAG integration tables — Vector
    "vdb_entities",
    "vdb_relationships",
    "vdb_chunks",
    # PathRAG integration tables — Graph
    "chunk_entity_relation_edges",
    "chunk_entity_relation_nodes",
]
for table in tables_to_drop:
    print(f"Dropping table '{table}'...")
    try:
        op = database.update_ddl([f"DROP TABLE IF EXISTS {table}"])
        op.result()
        print(f"  Done.")
    except Exception as e:
        print(f"  Skipped: {e}")

# Clean up temp working directory
if 'work_dir' in dir() and os.path.exists(work_dir):
    shutil.rmtree(work_dir, ignore_errors=True)
    print(f"\nRemoved temp dir: {work_dir}")

print("\n[OK] Cleanup completed")

---
## References

- [Cloud Spanner Documentation](https://cloud.google.com/spanner/docs)
- [Spanner Graph Overview](https://cloud.google.com/spanner/docs/graph/overview)
- [Spanner GQL Reference](https://cloud.google.com/spanner/docs/reference/standard-sql/graph-query-statements)